# CelebA Multi-label Classification - CPU Optimized (50k Subset)

## Project Overview
This notebook is optimized for training on a **CPU-only** setup or a local workstation with limited resources. It uses a **50,000 image subset** for faster experimentation.

### Optimizations for CPU:
- ✅ **Subset Training**: Limited to 50,000 images.
- ✅ **Stability**: `NUM_WORKERS=0` to avoid multiprocessing overhead/issues on Windows.
- ✅ **Precision**: 128x128 resolution for balance between speed and features.
- ✅ **Monitoring**: Comprehensive metrics (Acc, F1, Recall).

### Stage 1 Pipeline
1. Train on 50K images.
2. Save best weights for evaluation.


In [1]:
import os
import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ── Config / Parameters ──
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[DEVICE] Running on: {DEVICE}")

IMAGE_DIR      = r"C:\MLA\img_align_celeba"
ATTR_PATH      = r"C:\MLA\drive-download-20260318T181243Z-1-001\list_attr_celeba.txt"
PARTITION_PATH = r"C:\MLA\drive-download-20260318T181243Z-1-001\list_eval_partition.txt"

IMAGE_SIZE     = 128      
BATCH_SIZE     = 64       
NUM_WORKERS    = 0        # Safest for Windows CPU
SUBSET_SIZE    = 50000

NUM_ATTRS               = 40
DROPOUT                 = 0.4
EPOCHS                  = 10      # Reduced epochs for CPU subset
LEARNING_RATE           = 1e-3
PREDICTION_THRESHOLD    = 0.4 

# Create directories
os.makedirs("outputs", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)


[DEVICE] Running on: cpu


## 1. Data Loading

In [2]:
def load_dataframes():
    print("[DATA] Loading files...")
    attr = pd.read_csv(ATTR_PATH, sep=r'\s+', header=1, index_col=0)
    attr.index.name = 'image_id'
    attr = ((attr + 1) // 2).reset_index()
    
    splits = pd.read_csv(PARTITION_PATH, sep=' ', header=None, names=['image_id', 'split'])
    df = splits.merge(attr, on='image_id')
    
    train_df = df[df['split'] == 0].drop('split', axis=1).iloc[:SUBSET_SIZE].reset_index(drop=True)
    val_df   = df[df['split'] == 1].drop('split', axis=1).reset_index(drop=True)
    test_df  = df[df['split'] == 2].drop('split', axis=1).reset_index(drop=True)
    
    print(f"[DATA] Subset training size: {len(train_df):,} images")
    return train_df, val_df, test_df

train_df, val_df, test_df = load_dataframes()

print("\n[WEIGHTS] Computing class weights...")
attr_cols = [c for c in train_df.columns if c != 'image_id']
y_train = train_df[attr_cols].values
pos_counts = y_train.sum(axis=0)
pos_weight = torch.tensor((len(y_train) - pos_counts) / (pos_counts + 1e-6), dtype=torch.float32)
pos_weight = torch.clamp(pos_weight, max=10.0).to(DEVICE)


[DATA] Loading files...
[DATA] Subset training size: 50,000 images

[WEIGHTS] Computing class weights...


## 2. Dataset and Metrics

In [4]:
def transform_fn(img, is_train=False):
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
    if is_train and random.random() < 0.5: img = ImageOps.mirror(img)
    arr = np.array(img, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1)
    return (tensor - 0.5) / 0.5

class CelebADataset(Dataset):
    def __init__(self, df, img_dir, is_train=False):
        self.df = df
        self.img_dir = img_dir
        self.is_train = is_train
        self.attr_cols = [c for c in df.columns if c != 'image_id']
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['image_id'])).convert('RGB')
        return transform_fn(img, self.is_train), torch.tensor(row[self.attr_cols].values.astype(float), dtype=torch.float32)

def compute_metrics(preds, labels):
    preds = (torch.sigmoid(preds) > PREDICTION_THRESHOLD).float()
    tp = (preds * labels).sum(dim=0)
    fp = (preds * (1 - labels)).sum(dim=0)
    fn = ((1 - preds) * labels).sum(dim=0)
    acc = (preds == labels).float().mean().item() * 100
    prec = (tp / (tp + fp + 1e-8)).mean().item() * 100
    rec = (tp / (tp + fn + 1e-8)).mean().item() * 100
    f1 = (2 * prec * rec / (prec + rec + 1e-8))
    return {'acc': acc, 'f1': f1, 'rec': rec}

train_loader = DataLoader(CelebADataset(train_df, IMAGE_DIR, True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(CelebADataset(val_df,   IMAGE_DIR, False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


## 3. Model and Training

In [5]:
def conv_b(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, 1, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2))
class SimpleCNN(nn.Module):
    def __init__(self, n=40, d=0.4):
        super().__init__()
        self.m = nn.Sequential(conv_b(3, 64), conv_b(64, 128), conv_b(128, 128), conv_b(128, 256), nn.AdaptiveAvgPool2d(1))
        self.fc = nn.Linear(256, n)
        self.drop = nn.Dropout(d)
    def forward(self, x):
        x = self.m(x).view(x.size(0), -1)
        return self.fc(self.drop(x))

model = SimpleCNN().to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\n[TRAIN] Beginning training loop...")
for epoch in range(EPOCHS):
    model.train()
    t_loss = 0.0
    for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
    
    model.eval()
    v_loss, v_acc, ps, ls = 0.0, 0.0, [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out = model(imgs)
            v_loss += criterion(out, lbls).item()
            ps.append(out.cpu()); ls.append(lbls.cpu())
    
    metrics = compute_metrics(torch.cat(ps), torch.cat(ls))
    print(f"Epoch {epoch+1:02d} | Train Loss: {t_loss/len(train_loader):.4f} | Val Loss: {v_loss/len(val_loader):.4f} | Val Acc: {metrics['acc']:.2f}% | F1: {metrics['f1']:.2f}%")
    torch.save(model.state_dict(), "checkpoints/best_model_cpu.pth")



[TRAIN] Beginning training loop...


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html